# Collectives

Eight GPUs have just finished a backward pass. Each is holding a different gradient,
computed from a different slice of the batch. Before the optimizer can run, all eight
need to agree on one thing: the average.

The obvious way to do that is to send everything to rank 0, let it add the numbers up,
and send the result back out. It works, and it wastes most of what you paid for.

![Only one of these scales](../assets/diagrams/scaling.png)

Rank 0's link carries every byte, twice. Every other GPU sits idle waiting its turn, and
it gets worse with every GPU you add. The ring on the right costs each GPU one message
out and one in, no matter how many there are. That is the difference between a cluster
that scales to eight and one that scales to eight thousand.

NCCL gives you six of these, and every parallelism strategy in this course is built from
them.

![Six operations, four ranks](../assets/diagrams/collectives-grid.png)

Where each one shows up later:

| Collective | Used by |
|---|---|
| all-reduce | data parallelism, tensor parallelism |
| all-gather | ZeRO-3 gathering parameters before a layer runs |
| reduce-scatter | ZeRO-2 scattering gradients after the backward pass |
| all-to-all | routing tokens to experts in an MoE |
| broadcast | pushing the initial weights to every rank |
| send / recv | handing activations between pipeline stages |

**In this notebook:** run all six live on NCCL, then prove the identity that the next
few modules depend on.

---
## How these run

A collective needs several processes, one per GPU, and a notebook is a single process.
So each demo gets written to a file with `%%writefile` and launched with
`torch.distributed.run`, which is the module behind the `torchrun` command.

We launch through `sys.executable` so the child processes use this project's virtual
environment rather than whatever `python` happens to be first on `PATH`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

SCRATCH = "scratch-collectives"

# Generated scripts land in their own directory. Safe to re-run.
here = Path.cwd()
base = here.parent if here.name == SCRATCH else here
work = base / SCRATCH
work.mkdir(exist_ok=True)
os.chdir(work)

MAX_DEMO_RANKS = 4   # four ranks keeps the printed tables readable
NUM_GPUS = torch.cuda.device_count()
WORLD_SIZE = min(NUM_GPUS, MAX_DEMO_RANKS)

print(f"gpus available : {NUM_GPUS}")
print(f"ranks for demos: {WORLD_SIZE}")
print(f"working in     : {work}")


def launch(script, world_size=WORLD_SIZE, port=29700):
    """Run `script` under torch.distributed.run and stream back its output."""
    cmd = [sys.executable, "-m", "torch.distributed.run",
           "--nproc_per_node", str(world_size),
           "--master_port", str(port),
           script]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        print("FAILED\n", result.stderr[-2000:])
        return
    print(result.stdout)

---
## All six, live

One script, one section per collective. Each rank starts with values tagged by its own
rank number, so you can see exactly where every value ends up.

NCCL implements the GPU collectives but not `scatter` and `gather`, so those two run on
the gloo backend over CPU tensors. That is not a real limitation: nothing on the hot
path of distributed training needs them.

In [ ]:
%%writefile collectives_demo.py
'''Run every collective and print each rank's buffer before and after.'''
import os

import torch
import torch.distributed as dist

CHUNK = 2   # elements per rank, so a full buffer is CHUNK * world_size


def table(label, tensor, rank, world_size):
    '''Gather to rank 0 and print one row per rank, so output is deterministic.'''
    gathered = [torch.zeros_like(tensor) for _ in range(world_size)]
    dist.all_gather(gathered, tensor)
    if rank != 0:
        return

    print(f"    {label}")
    for r, t in enumerate(gathered):
        print(f"      rank{r}: {t.tolist()}")


def section(title, rank):
    if rank == 0:
        print(f"\n=== {title} ===")


def main():
    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")

    # Integers, so every comparison in this notebook is exact.
    full = torch.arange(CHUNK * world_size, dtype=torch.int64, device=device) + rank * 100
    scalar = torch.full((CHUNK,), rank + 1, dtype=torch.int64, device=device)

    # --- broadcast: rank 0's buffer overwrites everyone else's ---
    section("broadcast (src=0)", rank)
    buf = scalar.clone()
    table("before", buf, rank, world_size)
    dist.broadcast(buf, src=0)
    table("after", buf, rank, world_size)

    # --- all_gather: every rank ends up with every rank's chunk ---
    section("all_gather", rank)
    table("before", scalar, rank, world_size)
    out = [torch.zeros_like(scalar) for _ in range(world_size)]
    dist.all_gather(out, scalar)
    if rank == 0:
        print(f"      concatenated: {torch.cat(out).tolist()}")

    # --- reduce: only the destination rank gets the sum ---
    section("reduce (sum, dst=0)", rank)
    buf = scalar.clone()
    table("before", buf, rank, world_size)
    dist.reduce(buf, dst=0, op=dist.ReduceOp.SUM)
    table("after", buf, rank, world_size)

    # --- all_reduce: every rank gets the sum ---
    section("all_reduce (sum)", rank)
    buf = scalar.clone()
    table("before", buf, rank, world_size)
    dist.all_reduce(buf, op=dist.ReduceOp.SUM)
    table("after", buf, rank, world_size)

    # --- reduce_scatter: sum, then each rank keeps only its own slice ---
    section("reduce_scatter (sum)", rank)
    table("before (full buffer)", full, rank, world_size)
    shard = torch.zeros(CHUNK, dtype=torch.int64, device=device)
    dist.reduce_scatter(shard, list(full.chunk(world_size)), op=dist.ReduceOp.SUM)
    table("after (one shard each)", shard, rank, world_size)

    # --- all_to_all: a transpose across ranks ---
    section("all_to_all", rank)
    table("before", full, rank, world_size)
    recv = torch.zeros_like(full)
    dist.all_to_all_single(recv, full)
    table("after", recv, rank, world_size)

    # --- scatter and gather need gloo: NCCL does not implement them ---
    gloo = dist.new_group(backend="gloo")

    section("scatter (src=0, gloo/cpu)", rank)
    src_chunks = [torch.full((CHUNK,), r + 1, dtype=torch.int64) for r in range(world_size)]
    got = torch.zeros(CHUNK, dtype=torch.int64)
    dist.scatter(got, src_chunks if rank == 0 else None, src=0, group=gloo)
    if rank == 0:
        print(f"      rank0 sent: {[c.tolist() for c in src_chunks]}")
    table("after", got.to(device), rank, world_size)

    section("gather (dst=0, gloo/cpu)", rank)
    mine = torch.full((CHUNK,), rank + 1, dtype=torch.int64)
    bucket = [torch.zeros(CHUNK, dtype=torch.int64) for _ in range(world_size)] if rank == 0 else None
    dist.gather(mine, bucket, dst=0, group=gloo)
    if rank == 0:
        print(f"      rank0 received: {[b.tolist() for b in bucket]}")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
launch("collectives_demo.py", port=29700)

---
## The identity that makes ZeRO work

Look at reduce-scatter and all-gather next to each other and something falls out.

![all-reduce = reduce-scatter + all-gather](../assets/diagrams/allreduce-identity.png)

An all-reduce *is* those two halves, back to back. Which raises the obvious question: if
a rank only needs its own slice, why is it paying for the second half?

In plain data parallelism it has no choice, because every rank updates every parameter
and so needs the whole summed gradient. But if each rank owns only a slice of the
optimizer state, it only needs that slice. Stop after the reduce-scatter. You have just
halved the gradient traffic and cut optimizer memory by the number of GPUs, and it cost
you nothing.

That is ZeRO-2, and it falls straight out of this one identity.

The check below uses integers, so `torch.equal` is an exact comparison rather than an
approximate one.

In [ ]:
%%writefile allreduce_identity.py
'''Prove that all_reduce equals reduce_scatter followed by all_gather.'''
import torch
import torch.distributed as dist

ELEMENTS_PER_RANK = 4
SEED = 1234


def main():
    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")

    # Integer data: all_reduce on int64 is exact, so a bitwise check is valid.
    torch.manual_seed(SEED + rank)
    size = ELEMENTS_PER_RANK * world_size
    data = torch.randint(0, 1000, (size,), dtype=torch.int64, device=device)

    # Path A: a single all_reduce.
    direct = data.clone()
    dist.all_reduce(direct, op=dist.ReduceOp.SUM)

    # Path B: reduce_scatter, then all_gather the shards back together.
    shard = torch.zeros(ELEMENTS_PER_RANK, dtype=torch.int64, device=device)
    dist.reduce_scatter(shard, list(data.clone().chunk(world_size)), op=dist.ReduceOp.SUM)

    shards = [torch.zeros_like(shard) for _ in range(world_size)]
    dist.all_gather(shards, shard)
    composed = torch.cat(shards)

    identical = torch.equal(direct, composed)

    # Every rank must agree, so reduce the boolean before reporting.
    agreement = torch.tensor([int(identical)], dtype=torch.int64, device=device)
    dist.all_reduce(agreement, op=dist.ReduceOp.SUM)

    if rank == 0:
        print(f"  all_reduce            : {direct.tolist()}")
        print(f"  reduce_scatter+gather : {composed.tolist()}")
        print(f"  bitwise identical     : {identical}")
        print(f"  ranks agreeing        : {agreement.item()}/{world_size}")
        assert agreement.item() == world_size, "identity failed on at least one rank"
        print("\n  PASS: all_reduce == reduce_scatter + all_gather")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
launch("allreduce_identity.py", port=29710)

---
## What each one costs

With a ring algorithm, every collective moves a fixed amount per rank no matter how many
ranks there are. Watch where the `2(N-1)/N` for all-reduce comes from: the data goes
round once to build the sums, and round again to hand them out.

![Ring all-reduce](../assets/diagrams/ring-allreduce.gif)

`D` is the buffer size in bytes and `N` the number of ranks.

| Collective | Bytes each rank sends |
|---|---|
| broadcast | `(N-1)/N * D` |
| all-gather | `(N-1)/N * D` |
| reduce-scatter | `(N-1)/N * D` |
| **all-reduce** | **`2(N-1)/N * D`** |
| all-to-all | `(N-1)/N * D` |

Two things follow, and both shape the rest of the course.

As `N` grows, `(N-1)/N` approaches 1 and the cost per rank stops growing. That is why
ring all-reduce scales to thousands of GPUs and sending everything to rank 0 does not.

And all-reduce costs exactly twice reduce-scatter, so anything that can stop halfway
halves its communication.

Next up: what it takes to actually hide one of these behind the math.